# Ellipse Parameter Estimation

Notebook highlighting approaches to ellipse parameter estimation. There are two standard ways to compute ellipse parameters:

- In sequence using `glomar_gridding.ellipse.EllipseBuilder.compute_params`
- In parallel using `glomar_gridding.ellipse.estimation.get_ellipse_params`. This approach pre-computes input training data for a batch of positions, over which a parallel loop is performed estimating ellipse parameters for each position in the batch in parallel.

In [1]:
import numpy as np

from glomar_gridding.io import load_array
from glomar_gridding.ellipse.model import EllipseModel
from glomar_gridding.ellipse.estimate import EllipseBuilder, get_ellipse_params

## Set-up

Parameters used for parallelisation used in the notebook:

- `NJOBS` will specify the number of threads used
- `BACKEND` is the `joblib.Parallel` backend
- `BATCH_SIZE` represents the number of points from to use in each batch of jobs for parallel looping.

In [2]:
NJOBS = 4
BACKEND = "loky"
BATCH_SIZE = NJOBS * 16

## Training Data

Load the data used for ellipse parameter estimation.

In [3]:
esa_anom = load_array(
    "./esa_cci_sst_5deg_monthly_1982-2022_03.nc", "sst_anomaly"
)

esa_anom = esa_anom.rename({"lat": "latitude", "lon": "longitude"})
esa_anom_data = np.ma.masked_greater(esa_anom.values, 1e5)
esa_anom_coords = esa_anom.coords

## Ellipse Builder

Construct an instance of `EllipseBuilder` class. This computes the correlation matrix of the input data, from which ellipse parameters will be estimated.

In [4]:
ellipse_builder = EllipseBuilder(esa_anom_data, esa_anom_coords)

Numerical error correction applied to correlation matrix diagonal With difference to 1: 1.1920928955078125e-07


## Ellipse Model

Define the ellipse model, and values used for predition (initial guesses, boundaries, maximum distance used for training data, and default values (for masked positions)).

In [5]:
ellipse_model = EllipseModel(
    anisotropic=True,
    rotated=True,
    physical_distance=True,
    v=1.5,
    unit_sigma=True,
)

default_values = [
    -999.9,  # lx
    -999.9,  # ly
    -999.9,  # theta
    -999.9,  # stdev
    -1,  # success
    -1,  # niter
]

# Init values set to HadCRUT5 defaults
# no prior distribution set around those value
GUESSES = [
    2000.0,
    2000.0,
    0,
]

# Uniformative prior of parameter range
BOUNDS = [
    (300.0, 30000.0),
    (300.0, 30000.0),
    (-0.5 * np.pi, 0.5 * np.pi),
]

# Max range for training data
MAX_DIST = 10_000.0

## Parallel

Using `glomar_gridding.ellipse.estimate.get_ellipse_params`

In [6]:
%%time
results_par = get_ellipse_params(
    ellipse_model,
    ellipse_builder,
    n_par_jobs=NJOBS,
    par_backend=BACKEND,
    max_distance=MAX_DIST,
    bounds=BOUNDS,
    guesses=GUESSES,
    default_value=default_values,
    batch_size=BATCH_SIZE,
)

CPU times: user 702 ms, sys: 82.4 ms, total: 785 ms
Wall time: 16.9 s


In [7]:
print(results_par)

<xarray.Dataset> Size: 125kB
Dimensions:               (latitude: 36, longitude: 72)
Coordinates:
  * latitude              (latitude) float32 144B -87.5 -82.5 ... 82.5 87.5
  * longitude             (longitude) float32 288B -177.5 -172.5 ... 172.5 177.5
Data variables:
    Lx                    (latitude, longitude) float64 21kB -999.9 ... -999.9
    Ly                    (latitude, longitude) float64 21kB -999.9 ... -999.9
    theta                 (latitude, longitude) float64 21kB -999.9 ... -999.9
    standard_deviation    (latitude, longitude) float64 21kB -999.9 ... -999.9
    qc_code               (latitude, longitude) float64 21kB -1.0 -1.0 ... -1.0
    number_of_iterations  (latitude, longitude) float64 21kB -1.0 -1.0 ... -1.0


## Sequence

In sequence using `glomar_gridding.ellipse.EllipseBuilder.compute_params` class method.

In [8]:
%%time
results = ellipse_builder.compute_params(
    default_value=default_values,
    matern_ellipse=ellipse_model,
    max_distance=MAX_DIST,
    guesses=GUESSES,
    bounds=BOUNDS,
)

CPU times: user 57.8 s, sys: 405 ms, total: 58.2 s
Wall time: 58.3 s


## Compare Results

Only difference should be `number_of_iterations`.

In [9]:
for p in results.data_vars:
    print(
        f"{p:<25} Match: {np.allclose(results[p], results_par[p], atol=1e-4)}"
    )

Lx                        Match: True
Ly                        Match: True
theta                     Match: True
standard_deviation        Match: True
qc_code                   Match: True
number_of_iterations      Match: False
